# ⚡ AlurCerita Studio — Multi-Engine High-Speed TTS Bridge (Google Colab)
### 🚀 3 Engine Suara Pilihan: Kokoro-82M, Nari Dia-1.6B & Coqui XTTS v2

Notebook ini menghubungkan AlurCerita Studio dengan 3 model TTS tercanggih langsung dari Google Colab GPU (Tesla T4):
1. **Kokoro-82M (Default)**: Ultra-cepat (0.1x realtime), sangat jernih, ringan & konsisten untuk narasi alur cerita panjang.
2. **Nari Dia-1.6B**: Model TTS 1.6 Miliar parameter berbasis dialog & narasi emosional. Mendukung efek non-verbal `(laughs)`, `(sighs)`, `(whispers)` dan multi-speaker `[S1]` / `[S2]`.
3. **Coqui XTTS v2**: Multi-bahasa Indonesia & kloning suara ekspresif.

Fitur Utama:
- **Cloudflare Anycast Tunnel (`trycloudflare.com`)**: Mengalirkan data audio melalui edge server global Cloudflare tanpa batas proxy Gradio.
- **Hardware MP3 Compression (FFmpeg)**: Mengonversi audio WAV mentah menjadi MP3 192k di GPU Colab dalam 0,05 detik (transfer 10x lebih cepat).

---
### 📋 Cara Menjalankan:
1. Pastikan Runtime menggunakan GPU: **Runtime** -> **Change runtime type** -> pilih **T4 GPU** (Gratis).
2. Klik **Runtime** -> **Run all** (atau `Ctrl + F9`).
3. Tunggu hingga cell terakhir selesai. Anda akan mendapatkan URL **Cloudflare Tunnel** (contoh: `https://xxxx.trycloudflare.com`).
4. Tempelkan URL tersebut ke menu **🎙️ Voiceover TTS** di AlurCerita Studio!


In [ ]:
# 1. Install Dependensi Kilat (Kokoro-82M, Nari Dia-1.6B, Coqui XTTS & Cloudflare Tunnel)
import os
# Menggunakan uv package manager: selesai dalam 5-10 detik
!pip install -q uv

# Hapus torchvision (mencegah circular import di Transformers saat memuat Albert Kokoro)
!pip uninstall -y -q torchvision

# Pasang paket utama, Kokoro, dependensi Dia (descript-audio-codec, safetensors, huggingface_hub) & Coqui TTS
!uv pip install --system -q fastapi uvicorn pydantic soundfile nest_asyncio kokoro "misaki[en]" safetensors huggingface_hub descript-audio-codec coqui-tts
# Pasang Dia TTS tanpa merusak dependensi torch Colab
!pip install -q --no-deps git+https://github.com/nari-labs/dia.git

# Download dan install Cloudflared Linux x64
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Cloudflared Tunnel & Seluruh Engine TTS (Kokoro, Dia & XTTS) berhasil dipasang!")


In [ ]:
# 2. Inisialisasi Model TTS (Kokoro-82M, Nari Dia-1.6B & Coqui XTTS)
import sys
import torch
import soundfile as sf
import io
import numpy as np

# Bypass ketergantungan torchvision pada Transformers (karena Kokoro & Dia adalah model audio murni)
sys.modules['torchvision'] = None
sys.modules['torchvision.transforms'] = None
sys.modules['torchvision.ops'] = None
try:
    import transformers.utils.import_utils as import_utils
    import_utils.is_torchvision_available = lambda: False
except Exception:
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"⚡ Menggunakan perangkat: {device} ({gpu_name})")

# 1. Inisialisasi Kokoro-82M Pipeline (Model Utama Ultra-Cepat & Jernih)
print("📦 Memuat Kokoro-82M ke GPU...")
from kokoro import KPipeline
kokoro_pipeline = KPipeline(lang_code="a")
print("✅ Kokoro-82M siap!")

# 2. Inisialisasi Nari Dia-1.6B (Text-to-Dialogue & Expressive TTS)
dia_model = None
def get_dia():
    global dia_model
    if dia_model is None:
        from dia.model import Dia
        print("📦 Memuat Nari Dia-1.6B ke GPU (T4 float16)...")
        dia_model = Dia.from_pretrained("nari-labs/Dia-1.6B", compute_dtype="float16")
        print("✅ Nari Dia-1.6B siap!")
    return dia_model

# Unduh dan muat Nari Dia-1.6B ke VRAM
print("📦 Menyiapkan Nari Dia-1.6B...")
try:
    get_dia()
except Exception as e:
    print(f"⚠️ Info: Dia akan dimuat saat pertama kali dipilih (detail: {e})")

# 3. Inisialisasi Coqui XTTS v2
xtts_model = None
def get_xtts():
    global xtts_model
    if xtts_model is None:
        try:
            from TTS.api import TTS
        except ImportError:
            try:
                from coqui_tts.api import TTS
            except ImportError:
                raise RuntimeError("Coqui TTS belum terpasang. Model Kokoro-82M atau Dia-1.6B siap digunakan.")
        print("📦 Memuat Coqui XTTS v2 ke GPU...")
        xtts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
        print("✅ Coqui XTTS v2 siap!")
    return xtts_model


In [ ]:
# 3. Generator Audio Cepat (Auto MP3 Compression via FFmpeg)
import subprocess
import tempfile
import os

def generate_audio_fast(text, model="kokoro", voice="am_michael", speed=1.0, output_format="mp3"):
    text = text.strip()
    if not text:
        raise ValueError("Teks kosong")
    
    speed_float = float(speed)
    
    # Buat file WAV mentah sementara
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as raw_wav:
        raw_wav_path = raw_wav.name

    try:
        if model == "dia":
            dia = get_dia()
            clean_text = text.strip()
            # Format prefix speaker jika belum ada tag dialog [S1] atau [S2]
            if not ("[S1]" in clean_text or "[S2]" in clean_text):
                prefix = "[S2] " if ("s2" in voice.lower() or "character" in voice.lower()) else "[S1] "
                clean_text = prefix + clean_text
            
            output_audio = dia.generate(clean_text)
            if torch.is_tensor(output_audio):
                output_audio = output_audio.detach().cpu().numpy()
            if hasattr(output_audio, 'ndim') and output_audio.ndim > 1:
                output_audio = output_audio.squeeze()
            sample_rate = getattr(dia, "sample_rate", 44100)
            sf.write(raw_wav_path, output_audio, sample_rate, format="WAV")

        elif model == "xtts":
            tts = get_xtts()
            speaker_choice = voice if voice in tts.speakers else "Damian Black"
            tts.tts_to_file(
                text=text,
                file_path=raw_wav_path,
                speaker=speaker_choice,
                language="id",
                speed=speed_float
            )

        else:
            voice_choice = voice if voice else "am_michael"
            generator = kokoro_pipeline(text, voice=voice_choice, speed=speed_float)
            audio_segments = []
            for _, _, audio in generator:
                audio_segments.append(audio)
            
            if not audio_segments:
                raise RuntimeError("Kokoro tidak menghasilkan audio.")
            full_audio = np.concatenate(audio_segments)
            sf.write(raw_wav_path, full_audio, 24000, format="WAV")

        # Jika format yang diminta MP3, kompres via FFmpeg (10x lebih cepat transfer!)
        if output_format.lower() == "mp3":
            with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as mp3_out:
                mp3_path = mp3_out.name
            
            # Konversi cepat ke MP3 192k
            cmd = ["ffmpeg", "-y", "-i", raw_wav_path, "-codec:a", "libmp3lame", "-b:a", "192k", mp3_path]
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            
            with open(mp3_path, "rb") as f:
                audio_bytes = f.read()
            if os.path.exists(mp3_path):
                os.remove(mp3_path)
            return audio_bytes, "audio/mpeg"
        else:
            with open(raw_wav_path, "rb") as f:
                audio_bytes = f.read()
            return audio_bytes, "audio/wav"
            
    finally:
        if os.path.exists(raw_wav_path):
            os.remove(raw_wav_path)

print("✅ Generator Audio Cepat (Kokoro-82M, Nari Dia-1.6B & Coqui XTTS) siap!")


In [ ]:
# 4. Jalankan FastAPI Server & Cloudflare High-Speed Tunnel
import threading
import time
import re
from fastapi import FastAPI, Response, HTTPException
from pydantic import BaseModel
import uvicorn

api_app = FastAPI(title="AlurCerita High-Speed TTS Bridge")

class TTSReq(BaseModel):
    text: str
    model: str = "kokoro"
    voice: str = "am_michael"
    speed: float = 1.0
    format: str = "mp3"  # "mp3" (10x lebih cepat) atau "wav"

@api_app.get("/api/health")
def health():
    gpu_str = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {
        "status": "online",
        "engine": "AlurCerita Studio High-Speed Colab Bridge",
        "tunnel": "Cloudflare Anycast (High Throughput)",
        "gpu": gpu_str,
        "models": ["kokoro", "dia", "xtts"],
        "formats": ["mp3", "wav"]
    }

@api_app.post("/api/tts")
def tts_endpoint(req: TTSReq):
    try:
        audio_bytes, media_type = generate_audio_fast(
            req.text,
            req.model,
            req.voice,
            req.speed,
            req.format
        )
        return Response(content=audio_bytes, media_type=media_type)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Jalankan Uvicorn FastAPI di port 8000 di background thread
def run_uvicorn():
    uvicorn.run(api_app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)
print("✅ FastAPI backend berjalan di http://127.0.0.1:8000")

# Jalankan Cloudflare Anycast Tunnel
print("🚀 Membuka Cloudflare High-Speed Anycast Tunnel...")
cf_cmd = ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"]
cf_proc = subprocess.Popen(cf_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

cf_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = cf_proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        cf_url = match.group(0)
        break

if cf_url:
    print("\n" + "="*65)
    print("🎉 KONEKSI CLOUDFLARE HIGH-SPEED BERHASIL DIBUAT!")
    print(f"👉 URL Anda (Salin ini ke AlurCerita Studio):\n   {cf_url}")
    print("="*65 + "\n")
    print("💡 Mengapa ini jauh lebih cepat dari Gradio?")
    print("   1. Jalur Anycast Cloudflare tidak memiliki limit antrian/proxy Gradio.")
    print("   2. Audio dikompresi ke MP3 192k di GPU Colab (ukuran file 85% lebih kecil).")
else:
    print("⚠️ Cloudflare tunnel tidak mendapatkan URL dalam 30 detik. Menjalankan fallback Gradio...")
    import gradio as gr
    def gradio_ui_fn(text, model, voice, speed):
        wav_bytes, _ = generate_audio_fast(text, model, voice, speed, "wav")
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
            f.write(wav_bytes)
            return f.name
    demo = gr.Interface(fn=gradio_ui_fn, inputs=["text", "text", "text", "number"], outputs="audio")
    demo.queue().launch(share=True)
